# ReValue AI · experimental detector training
Run in a **private** Colab notebook using **Runtime → Change runtime type → T4 GPU** on the free tier. If free GPU quota is unavailable, defer training. Outputs include private photos and reviewer metadata. Keep Drive access restricted and clear notebook outputs before sharing.

This notebook trains two detectors from official COCO weights under one recipe. The original ReValue checkpoint is a fixed comparator. Start with a frozen export prepared according to the training handoff. An unlabelled pool alone will stop at preflight.

## 1. Inspect the runtime

In [ ]:
import os, sys, subprocess, hashlib, json, shutil, tempfile
from pathlib import Path
import torch

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
print("Python", sys.version, "PyTorch", torch.__version__, "CUDA", torch.version.cuda,
      "GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable")

## 2. Install the detector and open the versioned project
Use the Colab-provided CUDA PyTorch build. The project revision below pins the training and evaluation code. Runtime package versions are recorded with the run.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
import ultralytics
assert ultralytics.__version__ == "8.4.215", "Restart session and rerun after changing package versions."
REV = "177590a1cf4dd06efc92d12549e383851e20d55b"
REPO = Path("/content/ReValue-colab-" + REV[:8])
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "training/colab-yolo11", "--single-branch",
                    "https://github.com/YANKEY-CODE/ReValue-AI.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REV], check=True)
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"]).strip()
os.environ["YOLO_AUTOINSTALL"] = "false"
from ultralytics import settings
settings.update({k: False for k in ("sync","wandb","mlflow","comet","clearml","neptune","hub","dvc") if k in settings})

## 3. Connect private Google Drive
Authorize your own Drive when Colab prompts. Put your reviewed export under `MyDrive/ReValue-FYP/frozen`. Use a new experiment folder when changing the data or training settings. Use one Colab session per folder.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PRIVATE = Path("/content/drive/MyDrive/ReValue-FYP")
FROZEN = PRIVATE / "frozen"
WORK = PRIVATE / "experiments/control-s17-v1"
WORK.mkdir(parents=True, exist_ok=True)

## 4. Optional licensed image intake
Set the switch to True to download the 95 CC BY 4.0 Broken Interfaces display crops into a separate review pool. This step records source and attribution and creates **no training labels**. Skip it when you already have an approved export. Other datasets need a permitted version and independent review before export.

In [ ]:
DOWNLOAD_REVIEW_POOL = False
if DOWNLOAD_REVIEW_POOL:
    source = Path("/content/broken-interfaces-source")
    revision = "37877e61c93e17975a8a5968baefad3dba0b3231"
    if not source.exists():
        subprocess.run(["git", "clone", "--no-checkout", "--filter=blob:none",
                        "https://github.com/MHCI/broken-interfaces.git", str(source)], check=True)
    subprocess.run(["git","-C",str(source),"sparse-checkout","set","--no-cone","/phone_cropped_display/","/README.md"],check=True)
    subprocess.run(["git","-C",str(source),"checkout","--detach",revision],check=True)
    subprocess.run([sys.executable,str(REPO/"fyp/benchmark/dataset.py"),"--root",str(PRIVATE/"broken-interfaces-review-pool"),
                    "import-broken",str(source)],check=True)
    # Reserve the project's deterministic blind practice group on first import.
    pool = PRIVATE/"broken-interfaces-review-pool"
    if not (pool/"practice/roster.json").is_file():
        rows = [json.loads(f.read_text()) for f in (pool/"records").glob("*.json")]
        if len(rows) != 95 or any(r["split"] != "pool" or r["annotation"] for r in rows):
            raise ValueError("Restore the original practice roster before using this modified pool.")
        subprocess.run([sys.executable,str(REPO/"fyp/benchmark/dataset.py"),"--root",str(pool),
                        "prepare-practice"],check=True)

## 5. Validate and organize the frozen data
Only the three manifest splits, named label files and named PNGs are copied into an isolated runtime directory. The inherited eligibility rules, cross-split identities, independent review, class order, file hashes, dimensions, negative coverage and annotation bounds are checked before fitting. Keep this input export read-only. Any validation error requires correcting the source export.

In [ ]:
from collections import Counter
if not (FROZEN/"manifest.json").is_file():
    raise FileNotFoundError(f"Missing reviewed export at {FROZEN}. Follow fyp/training/README.md; a review pool is insufficient.")
sys.path.insert(0,str(REPO/"fyp/benchmark"))
from dataset import key, digest
manifest = json.loads((FROZEN/"manifest.json").read_text())
DATA = Path("/content") / ("revalue-frozen-" + digest(FROZEN/"manifest.json")[:16])
# A fresh session always restages. Within a session, prepare() rechecks staged hashes.
if not DATA.exists():
    with tempfile.TemporaryDirectory(dir="/content") as tmp:
        stage = Path(tmp)/"frozen"
        stage.mkdir()
        shutil.copyfile(FROZEN/"manifest.json",stage/"manifest.json")
        for split in ("train","val","test"):
            for folder, suffix in (("images",".png"),("labels",".txt")):
                (stage/folder/split).mkdir(parents=True)
                for record in manifest["splits"][split]:
                    rel = Path(folder)/split/(key(record["id"])+suffix)
                    shutil.copyfile(FROZEN/rel,stage/rel)
        stage.rename(DATA)
def pipeline(action, *args):
    subprocess.run([sys.executable, str(REPO/"fyp/training/pipeline.py"), action,
                    "--dataset",str(DATA),"--work",str(WORK),*map(str,args)],check=True)
pipeline("prepare")
for split, rows in manifest["splits"].items():
    print(split, len(rows), "images;", Counter(b["label"] for r in rows for b in r["annotation"]["boxes"]),
          "reviewed clean:",sum(r["annotation"]["kind"]=="clean" for r in rows))
environment = WORK / "packages.txt"
if not environment.exists():
    environment.write_bytes(subprocess.check_output([sys.executable,"-m","pip","freeze"]))

## 6. Inspect human annotations on training examples
The rectangles below come only from approved training annotations. Stop if a box, view or class is wrong. Correct the private source and freeze a new version.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
rows = manifest["splits"]["train"]
examples = []
for category in ("generic_crack","scratch","clean"):
    r = next(r for r in rows if r["annotation"]["kind"]==category or
             any(b["label"]==category for b in r["annotation"]["boxes"]))
    examples.append(r)
fig, axes = plt.subplots(1,3,figsize=(14,5))
for ax,r in zip(axes,examples):
    with Image.open(DATA/"images/train"/(r["id"]+".png")) as im: ax.imshow(im)
    ax.set_title(r["view"]+" · "+r["annotation"]["kind"]); ax.axis("off")
    for b in r["annotation"]["boxes"]:
        x,y,X,Y=b["xyxy"]; color="orange" if b["label"]=="generic_crack" else "cyan"
        ax.add_patch(Rectangle((x,y),X-x,Y-y,fill=False,edgecolor=color,linewidth=2))
        ax.text(x,y,b["label"],backgroundcolor="black",color=color)
plt.tight_layout(); plt.show()

## 7. Validate the original detector and fetch official initialization weights
Baseline validation uses the validation split. The test split remains reserved for the final comparison. Confirm that the runtime has a GPU before beginning either fit.

In [ ]:
pipeline("baseline")
import pandas as pd
display(pd.read_csv(WORK/"baseline-val/per_class.csv"))
if not torch.cuda.is_available():
    raise RuntimeError("Select a free T4 GPU, reconnect and rerun the notebook.")
import urllib.request
sys.path.insert(0,str(REPO/"fyp/training"))
from pipeline import INITIAL_SHA
weights = WORK/"initial"
weights.mkdir(exist_ok=True)
def initial(scale):
    filename=f"yolo11{scale}.pt"
    path=weights/filename
    if not path.exists():
        tmp=path.with_suffix(".partial")
        urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v8.4.0/"+filename,tmp)
        if digest(tmp)!=INITIAL_SHA[scale]: raise ValueError("Official release checksum mismatch.")
        tmp.replace(path)
    if digest(path)!=INITIAL_SHA[scale]: raise ValueError("Cached initialization hash mismatch.")
    return path

## 8. Train experimental YOLO11n
100 epochs, 640 px, batch 8, AdamW, seed 17, no geometric or appearance augmentation. The recipe and data hash are recorded. Drive receives last.pt each epoch, periodic backups and the best validation checkpoint.

In [ ]:
pipeline("train","--scale","n","--initial",initial("n"))

## 9. Train experimental YOLO11s
Use the identical training budget and split. After a Colab disconnect, reconnect, mount the same Drive and rerun. Completed fits are verified and skipped. An unfinished run resumes from its own last.pt.

In [ ]:
pipeline("train","--scale","s","--initial",initial("s"))

## 10. Inspect training graphs and validation confusion matrices
Use validation results to choose settings. Keep subsequent test results out of threshold and architecture tuning.

In [ ]:
from IPython.display import Image as DisplayImage, display
for scale in ("n","s"):
    run=WORK/f"yolo11{scale}-control"
    for file in ("results.png","confusion_matrix.png"):
        print(run.name, file)
        display(DisplayImage(filename=str(run/file),width=760))

## 11. Compare all three detectors on the same frozen test set
CPU, two threads, batch one, 640 px, FP32, NMS IoU 0.70. Operating confidence 0.25 and match IoU 0.50. AP uses scores down to 0.001. Warm-up precedes interleaved timed predictions. Private error records contain example IDs and box coordinates.

In [ ]:
pipeline("compare")
results=WORK/"final-test"
metrics=pd.read_csv(results/"per_class.csv")
summary=json.loads((results/"summary.json").read_text())
display(metrics)
display(pd.DataFrame(summary["models"]).T)
ax=metrics.pivot(index="class",columns="model",values="recall").plot.bar(ylim=(0,1),rot=0)
ax.set_ylabel("Held-out recall at confidence 0.25")
plt.tight_layout(); plt.savefig(results/"held-out-recall.png",dpi=150); plt.show()
print("Private FP and missed detections:",results/"private_errors.json")

## 12. Download the experimental checkpoints
This archive contains the two experimental best weights, their recipe records and completion hashes. Keep photos, plots and per-image failures private. Review crack recall together with scratch recall, FP on clean phones and CPU latency before recommending a model.

In [ ]:
from google.colab import files
from zipfile import ZipFile
bundle=WORK/"experimental-checkpoints.zip"
with ZipFile(bundle,"w") as z:
    for scale in ("n","s"):
        name=f"yolo11{scale}-control"
        for path in (WORK/name/"weights/best.pt",WORK/(name+".json"),WORK/(name+"-complete.json")):
            z.write(path,path.relative_to(WORK))
files.download(str(bundle))